# Icelines

Comparison of `DustPy` (full coagulation) and `TriPoDPy` for a disk in which the fragmentation velocity
changes at the temperatures 150 K, 80 K and 44 K, mimicking ice lines (no gas evolution).

Before running this notebook, run both simulations:

* `python Icelines.py` in this folder (TriPoDPy, writes `data/`)
* `python Icelines.py` inside `Dustpy_setups/` (DustPy, writes `Dustpy_setups/data/`)

Both setups write to a folder called `data`, so run the DustPy setup from `Dustpy_setups/`
to avoid overwriting the TriPoDPy output.

In [ ]:
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import dustpy as dp
from tripodpy.utils import get_q, get_size_distribution

au = dp.constants.au
yr = dp.constants.year

# figures are saved here
plot_dir = Path("plots/iceline")
plot_dir.mkdir(parents=True, exist_ok=True)

## Read data

In [ ]:
dp_path = Path("Dustpy_setups/data")  # DustPy output
tp_path = Path("data")                # TriPoDPy output

In [ ]:
def read_dustpy(path):
    """Read a DustPy run and compute the quantities compared below."""
    reader = dp.hdf5writer()
    reader.datadir = str(path)
    data = reader.read.all()

    a = data.dust.a[0, 0]
    Sigma_d = data.dust.Sigma

    # a_max: particle size below which 99% of the dust mass is contained
    cum_mass = np.cumsum(Sigma_d, axis=-1)
    a_max = a[np.argmax(cum_mass / cum_mass[..., -1:] >= 0.99, axis=-1)]

    return SimpleNamespace(
        t=np.ravel(data.t),
        r=data.grid.r[0],
        a=a,
        Sigma_g=data.gas.Sigma,
        Sigma_d=Sigma_d,
        eps=data.dust.eps,
        a_1=(Sigma_d * a).sum(-1) / Sigma_d.sum(-1),  # mass-averaged particle size
        a_max=a_max,
    )


def read_tripodpy(path, a):
    """Read a TriPoDPy run and reconstruct its power-law size distribution
    on the particle size grid `a` (the DustPy grid)."""
    reader = dp.hdf5writer()
    reader.datadir = str(path)
    data = reader.read.all()

    Sigma_d = data.dust.Sigma  # (n_t, n_r, 2): small and large population
    a_max = data.dust.a[..., 4]

    # interfaces chosen such that the bin centres coincide with `a`
    A = np.mean(a[1:] / a[:-1])
    with np.errstate(divide="ignore", invalid="ignore"):
        q = get_q(Sigma_d, data.dust.s.min, a_max)
        _, _, Sigma_d_rec = get_size_distribution(
            Sigma_d.sum(-1),
            a_max,
            q=q,
            na=len(a),
            agrid_min=2 / (A + 1) * a[0],
            agrid_max=2 * A / (A + 1) * a[-1],
        )

    return SimpleNamespace(
        t=np.ravel(data.t),
        r=data.grid.r[0],
        a=a,
        Sigma_g=data.gas.Sigma,
        Sigma_d=Sigma_d_rec,
        eps=data.dust.eps,
        a_1=data.dust.a[..., 2],
        a_max=a_max,
    )

In [ ]:
d_d = read_dustpy(dp_path)
d_t = read_tripodpy(tp_path, d_d.a)

for name, d in [("DustPy", d_d), ("TriPoDPy", d_t)]:
    print(f"{name:9s}: {len(d.t)} snapshots, t_end = {d.t[-1] / yr:.3g} yr")

# line styles for the comparison plots
styles = {"DustPy": dict(ls="-", c="k"), "TriPoDPy": dict(ls="--", c="C2")}

## Plots

### 2D distribution plot

In [ ]:
f, axs = plt.subplots(1, 3, figsize=(10, 5), gridspec_kw={'width_ratios': [1, 1, 0.07], 'wspace': 0.05})

vmin = 1e-6
vmax = 1e-1
time = 0.5e5*yr

for ax, d in zip(axs, [d_d, d_t]):

    it = np.abs(d.t - time).argmin()

    pcm = ax.pcolormesh(d.r / au, d.a, d.Sigma_d[it, ...].T, norm=LogNorm(vmin, vmax), cmap='magma')
    con = ax.contour(d.r / au, d.a, d.Sigma_d[it, ...].T, np.geomspace(vmin, vmax, int(np.log10(vmax/vmin))+1), norm=LogNorm())
    ax.set_facecolor(plt.get_cmap('magma')(0.0))  # empty bins (masked by LogNorm) drawn like values below vmin
    ax.loglog()
    ax.set_xlabel('radius [au]')
    ax.set_title(f't = {d.t[it] / yr:.6g} yr')

axs[0].set_ylabel('particle size [cm]')
for ax in axs[1:]:
    ax.set_yticklabels('')

cb1 = f.colorbar(pcm, cax=axs[-1])
cb2 = f.colorbar(con, cax=axs[-1])
cb2.lines[0].set_linewidth(2)
plt.savefig(plot_dir / f"size_dist_{it}")

### Surface densities and $a_1$

In [ ]:
f, axs = plt.subplots(1, 2, figsize=(20/3, 3))

time = 0.5e5*yr

for ax, d, title in zip(axs, [d_d, d_t], ['DustPy', 'TriPoDPy']):
    it = np.abs(d.t - time).argmin()
    ax.loglog(d.r / au, d.Sigma_g[it], label=r"$\Sigma_g$")
    ax.loglog(d.r / au, d.Sigma_g[it] * d.eps[it], label=r"$\Sigma_{d}$")
    ax.loglog(d.r / au, d.a_1[it], 'k--', label="$a_1$")
    ax.set_title(title)
    ax.set_ylim(1e-6, 1e3)
axs[0].legend();

In [ ]:
f, axs = plt.subplots(1, 4, figsize=(10/3*4, 3), dpi=150)

time = 0.5e5*yr

for name, d in [("DustPy", d_d), ("TriPoDPy", d_t)]:
    it = np.abs(d.t - time).argmin()
    print(name, d.t[it]/yr)

    axs[0].loglog(d.r / au, d.Sigma_g[it], label=name, **styles[name])
    axs[1].loglog(d.r / au, d.Sigma_d[it].sum(-1), **styles[name])
    axs[2].loglog(d.r / au, d.a_max[it], **styles[name])
    axs[3].loglog(d.r / au, d.a_1[it], **styles[name])

axs[0].set_title(r"$\Sigma_g$")
axs[1].set_title(r"$\Sigma_{d,tot}$")
axs[2].set_title(r"$a_{max}$")
axs[3].set_title(r"$a_{1}$")
axs[0].set_ylim(1e-6, 2e3)
axs[1].set_ylim(1e-8, 2e2)
axs[3].set_ylim(1e-5, 2e1)

f.suptitle("Time %.3f" % (d.t[it]/yr/1e6), y=1.05)
axs[0].legend()
plt.savefig(plot_dir / f"typical_sizes_{it}")

In [ ]:
fg = plt.figure()

time = 1e5*yr
r = 5 * au
ax = plt.gca()
ax.text(0.05, 0.9, "r = %.1f Au \nt = %.2f Myr" % (r/au, time/1e6/yr), transform=ax.transAxes)
for name, d in [("DustPy", d_d), ("TriPoDPy", d_t)]:
    it = np.abs(d.t - time).argmin()
    ir = np.abs(d.r - r).argmin()
    plt.plot(d.a, d.Sigma_d[it, ir, :], label=name, **styles[name])

plt.xscale("log")
plt.yscale("log")
plt.legend()
plt.ylim(1e-10)
plt.xlim(5e-5)
plt.savefig(plot_dir / f"local_dist_{it}_{ir}")